# Step 1 — Group Images by Slide ID

Parse all BreakHis `.png` filenames, extract slide ID / magnification / class / subtype,
and group images by slide. This is the foundation for slide-level splitting (no leakage).

### Setup and data path

Import dependencies and set the data root. Flip `USE_COLAB = True` when running in Google Colab — it will mount Google Drive automatically. Locally, it expects `BreaKHis_v1/` in the working directory.

In [2]:
import os
import re
import pandas as pd
from pathlib import Path
from collections import Counter

# ---------- toggle: Colab (Google Drive) vs local ----------
USE_COLAB = False

if USE_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    DATA_ROOT = Path("/content/drive/MyDrive/BreaKHis_v1")
else:
    DATA_ROOT = Path("BreaKHis_v1")

assert DATA_ROOT.exists(), f"Data not found at {DATA_ROOT}"

### Parse every filename into structured fields

Each BreakHis filename follows the format `SOB_{B|M}_{TYPE}-{YEAR}-{PATIENT_ID}-{MAG}-{SEQ}.png`. The regex extracts:
- **label** — benign (`B`) or malignant (`M`)
- **subtype** — tumor subtype code (e.g. `DC` = ductal carcinoma, `TA` = tubular adenoma)
- **slide_id** — `YEAR-PATIENT_ID` combination, the unique slide identifier
- **magnification** — 40, 100, 200, or 400
- **seq** — image sequence number within that slide/magnification

Any filename that doesn't match the expected pattern is collected in `unparsed` for inspection.

In [3]:
FILENAME_RE = re.compile(
    r"^SOB_([BM])_(\w+)-(\d+-\w+)-(\d+)-(\d+)\.png$"
)

records = []
unparsed = []

for png in DATA_ROOT.rglob("*.png"):
    m = FILENAME_RE.match(png.name)
    if not m:
        unparsed.append(png.name)
        continue
    class_code, subtype_code, slide_id, mag, seq = m.groups()
    label = "benign" if class_code == "B" else "malignant"
    records.append({
        "path": str(png),
        "filename": png.name,
        "label": label,
        "subtype": subtype_code,
        "slide_id": slide_id,
        "magnification": int(mag),
        "seq": int(seq),
    })

df = pd.DataFrame(records)
print(f"Parsed {len(df)} images, {len(unparsed)} unparsed")
if unparsed:
    print("Unparsed samples:", unparsed[:10])

Parsed 7909 images, 0 unparsed


### Preview the parsed dataframe

Quick sanity check — each row is one image with its extracted metadata.

In [4]:
df.head(10)

,path,filename,label,subtype,slide_id,magnification,seq
0,BreaKHis_v1/histology_slides/breast/benign/SOB...,SOB_B_A-14-29960CD-200-013.png,benign,A,14-29960CD,200,13
1,BreaKHis_v1/histology_slides/breast/benign/SOB...,SOB_B_A-14-29960CD-200-003.png,benign,A,14-29960CD,200,3
2,BreaKHis_v1/histology_slides/breast/benign/SOB...,SOB_B_A-14-29960CD-200-008.png,benign,A,14-29960CD,200,8
3,BreaKHis_v1/histology_slides/breast/benign/SOB...,SOB_B_A-14-29960CD-200-010.png,benign,A,14-29960CD,200,10
4,BreaKHis_v1/histology_slides/breast/benign/SOB...,SOB_B_A-14-29960CD-200-015.png,benign,A,14-29960CD,200,15
5,BreaKHis_v1/histology_slides/breast/benign/SOB...,SOB_B_A-14-29960CD-200-005.png,benign,A,14-29960CD,200,5
6,BreaKHis_v1/histology_slides/breast/benign/SOB...,SOB_B_A-14-29960CD-200-012.png,benign,A,14-29960CD,200,12
7,BreaKHis_v1/histology_slides/breast/benign/SOB...,SOB_B_A-14-29960CD-200-002.png,benign,A,14-29960CD,200,2
8,BreaKHis_v1/histology_slides/breast/benign/SOB...,SOB_B_A-14-29960CD-200-007.png,benign,A,14-29960CD,200,7
9,BreaKHis_v1/histology_slides/breast/benign/SOB...,SOB_B_A-14-29960CD-200-014.png,benign,A,14-29960CD,200,14


### Group images by slide

Aggregate all images belonging to the same slide. For each slide, record:
- Its class label (benign/malignant) and subtype
- Total number of images across all magnifications
- Which magnification levels are present

This is the grouping that all downstream splits must respect — images from the same slide must never end up in both train and test.

In [5]:
slides = df.groupby("slide_id").agg(
    label=pd.NamedAgg(column="label", aggfunc="first"),
    subtype=pd.NamedAgg(column="subtype", aggfunc="first"),
    n_images=pd.NamedAgg(column="filename", aggfunc="count"),
    magnifications=pd.NamedAgg(column="magnification", aggfunc=lambda x: sorted(x.unique().tolist())),
).reset_index()

print(f"{len(slides)} unique slides")
print(f"\nSlides per class:")
print(slides["label"].value_counts())
print(f"\nSlides per subtype:")
print(slides["subtype"].value_counts())
slides.head(10)

81 unique slides

Slides per class:
label
malignant    57
benign       24
Name: count, dtype: int64

Slides per subtype:
subtype
DC    37
F     10
MC     9
TA     7
PC     6
LC     5
A      4
PT     3
Name: count, dtype: int64


,slide_id,label,subtype,n_images,magnifications
0,14-10147,malignant,MC,64,"[40, 100, 200, 400]"
1,14-10926,malignant,DC,39,"[40, 100, 200, 400]"
2,14-11031,malignant,DC,60,"[40, 100, 200, 400]"
3,14-11520,malignant,DC,98,"[40, 100, 200, 400]"
4,14-11951,malignant,DC,100,"[40, 100, 200, 400]"
5,14-12204,malignant,LC,103,"[40, 100, 200, 400]"
6,14-12312,malignant,DC,146,"[40, 100, 200, 400]"
7,14-12465,malignant,PC,74,"[40, 100, 200, 400]"
8,14-12773,malignant,MC,90,"[40, 100, 200, 400]"
9,14-13200,benign,TA,65,"[40, 100, 200, 400]"


### Class × magnification cross-tab

Shows how images distribute across benign/malignant and the four magnification levels. Useful for spotting class imbalance per zoom level.

In [6]:
mag_counts = df.groupby(["label", "magnification"]).size().unstack(fill_value=0)
print("Images per class × magnification:")
mag_counts

Images per class × magnification:


magnification,40,100,200,400
label,,,,
benign,625,644,623,588
malignant,1370,1437,1390,1232


### Check for missing magnification levels

Verify that every slide has all four magnifications (40X, 100X, 200X, 400X). Slides with missing levels would need special handling during model-specific training.

In [7]:
missing_mags = slides[slides["magnifications"].apply(lambda m: set(m) != {40, 100, 200, 400})]
if len(missing_mags) == 0:
    print("All slides have all 4 magnification levels.")
else:
    print(f"{len(missing_mags)} slides missing magnification levels:")
    print(missing_mags)

All slides have all 4 magnification levels.


### Images-per-slide distribution

Summary statistics on how many images each slide contributes. Large variance here means some slides dominate the dataset — important context for stratified splitting.

In [8]:
print(f"Images per slide — min: {slides['n_images'].min()}, "
      f"max: {slides['n_images'].max()}, "
      f"median: {slides['n_images'].median():.0f}")
slides["n_images"].describe()

Images per slide — min: 38, max: 246, median: 90


count     81.000000
mean      97.641975
std       41.624304
min       38.000000
25%       64.000000
50%       90.000000
75%      125.000000
max      246.000000
Name: n_images, dtype: float64

### Export catalogs

Save two CSVs to `data/`:
- `image_catalog.csv` — one row per image with all parsed fields (used by training scripts)
- `slide_summary.csv` — one row per slide with aggregated stats (used for splitting in step 2)

In [ ]:
out_dir = Path("data")
out_dir.mkdir(exist_ok=True)

df.to_csv(out_dir / "image_catalog.csv", index=False)
slides.to_csv(out_dir / "slide_summary.csv", index=False)
print(f"Saved {out_dir}/image_catalog.csv and {out_dir}/slide_summary.csv")